# 📸 Realistic NSFW Image Generator
### Powered by LUSTIFY v2.0 · No Login Required · Colab T4 GPU

---

## ✅ Run in this exact order:

| # | Cell | What happens |
|---|------|--------------|
| 1 | **GPU Check** | Confirms T4 is attached |
| 2 | **Install** | Installs only diffusers — then **auto-restarts kernel** |
| 3 | **Verify** | Run after restart — confirms all imports work |
| 4 | **Load Model** | Downloads ~6.5 GB first time, then cached |
| 5 | **Launch UI** | Opens the image generator |

> ⚠️ After Cell 2, a **"Session crashed"** popup appears — this is **normal**.
> Click OK, then continue from **Cell 3**. Do NOT re-run Cell 2.

---

**Model:** LUSTIFY v2.0 (SDXL-based photorealistic NSFW checkpoint)  
**Prompting:** Natural language (full sentences work great — no special tags needed)  
**Speed:** ~60–90 seconds per image on T4  
**Login required:** None ✅

---

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 of 5 — GPU Check                                    ║
# ║  Must show a GPU (Tesla T4).                                 ║
# ║  If nothing shows: Runtime → Change runtime type →          ║
# ║  T4 GPU → Save, then reconnect and re-run.                  ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, sys

r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)

if r.returncode == 0 and r.stdout.strip():
    print(f'✅ GPU : {r.stdout.strip()}')
    print(f'   Python: {sys.version.split()[0]}')
    print()
    print('GPU confirmed — ready to proceed to Cell 2!')
else:
    print('❌ No GPU detected!')
    print('   Fix: Runtime → Change runtime type → T4 GPU → Save')
    raise SystemExit('GPU required. Please enable T4 GPU.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 of 5 — Install + Restart Kernel                     ║
# ║                                                              ║
# ║  ONLY installs: diffusers, accelerate, safetensors           ║
# ║  Does NOT touch: torch, transformers, gradio                 ║
# ║  (those are pre-installed in Colab at correct versions)      ║
# ║  Does NOT install xformers — causes torchvision crash        ║
# ║                                                              ║
# ║  Kernel restarts automatically at the end.                   ║
# ║  "Session crashed" popup is NORMAL — click OK.               ║
# ║  Then run Cell 3 → 4 → 5. Do NOT re-run this cell.         ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, sys

def pip(*args):
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + list(args),
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f'⚠️  pip warning for {args}: {r.stderr[-300:] if r.stderr else "(none)"}')
    return r.returncode

print('📦 Installing packages...')
print('   (Only diffusers, accelerate, safetensors — everything else already in Colab)')
print('─' * 62)

pip('diffusers')      # latest — fixes all FLAX_WEIGHTS_NAME / import errors
pip('accelerate')     # required by diffusers pipeline
pip('safetensors')    # required to load .safetensors model files

print('─' * 62)
print('✅ Packages installed!')
print()
print('🔄 Restarting kernel so Python loads the new diffusers...')
print('   "Session crashed" popup = NORMAL → click OK')
print('   Then continue from Cell 3.')

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 of 5 — Verify (run AFTER kernel restart)            ║
# ║  Checks all imports before wasting time on model download.  ║
# ║  All lines must print ✅ before continuing to Cell 4.       ║
# ╚══════════════════════════════════════════════════════════════╝

import sys
print(f'Python : {sys.version.split()[0]}')
print()

# ── torch + CUDA ───────────────────────────────────────────────
import torch
print(f'torch         : {torch.__version__}')
cuda_ok = torch.cuda.is_available()
print(f'CUDA available: {cuda_ok}')
if not cuda_ok:
    raise SystemExit('❌ CUDA not available. Make sure T4 GPU is selected.')
print(f'GPU name      : {torch.cuda.get_device_name(0)}')
print(f'VRAM total    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print()

# ── transformers ───────────────────────────────────────────────
import transformers
print(f'transformers  : {transformers.__version__}')

# ── diffusers (most critical) ──────────────────────────────────
import diffusers
print(f'diffusers     : {diffusers.__version__}')

# ── gradio ─────────────────────────────────────────────────────
import gradio
print(f'gradio        : {gradio.__version__}')
print()

# ── Test exact pipeline + scheduler imports ────────────────────
print('Testing imports...')
from diffusers import StableDiffusionXLPipeline
print('  ✅ StableDiffusionXLPipeline')

from diffusers import DPMSolverMultistepScheduler
print('  ✅ DPMSolverMultistepScheduler  (DPM++ 2M SDE Karras)')

print()
print('=' * 55)
print('✅  ALL CHECKS PASSED — safe to run Cell 4!')
print('=' * 55)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 of 5 — Load Model                                   ║
# ║  Model: LUSTIFY v2.0 (SDXL photorealistic NSFW)             ║
# ║  First run: downloads ~6.5 GB (5–10 min)                    ║
# ║  Later runs: loads from cache (~60 sec)                     ║
# ║  No HuggingFace login required.                             ║
# ╚══════════════════════════════════════════════════════════════╝

import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

# LUSTIFY v2.0:
# - SDXL 1.0 base fine-tuned for photorealistic + NSFW content
# - Fully public repo, no token required
# - Understands BOTH natural language AND tag-style prompts
# - Recommended sampler by model author: DPM++ 2M SDE / DPM++ 3M SDE, Exponential/Karras
MODEL_ID = 'TheImposterImposters/LUSTIFY-v2.0'

print(f'📥 Loading: {MODEL_ID}')
print('First run downloads ~6.5 GB — please wait...')
print('─' * 62)

# Load in float16 — fits in T4 VRAM (15 GB)
# NOTE: No variant="fp16" — this repo doesn't publish fp16 variant
#       files, so that argument would throw a 404 error.
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)

# DPM++ 2M SDE Karras — the model author's recommended scheduler.
# In diffusers this is DPMSolverMultistepScheduler with:
#   algorithm_type = 'sde-dpmsolver++'  → SDE variant
#   use_karras_sigmas = True             → Karras noise schedule
#   euler_at_final = True                → prevents artifacts at low step counts on SDXL
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type='sde-dpmsolver++',
    use_karras_sigmas=True,
    euler_at_final=True,
)

# Move model to GPU
pipe = pipe.to('cuda')

# Reduce VRAM usage — no extra packages, no conflicts
# Intentionally NOT using xformers — overwrites torch and breaks
# torchvision on Colab (operator torchvision::nms does not exist)
pipe.enable_attention_slicing()

# SDXL has no safety_checker attribute — intentionally not set

print('─' * 62)
print('✅ Model loaded and ready!')
used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'🖥️  VRAM used : {used:.2f} GB')
print(f'🖥️  VRAM free : {total - used:.2f} GB')
print(f'🖥️  VRAM total: {total:.2f} GB')
print()
print('✅ Ready — run Cell 5 to open the image generator!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 of 5 — Launch Image Generator UI                    ║
# ║  The interface appears below this cell after running.       ║
# ║  A public share link (72 hr) is also printed.               ║
# ╚══════════════════════════════════════════════════════════════╝

import gradio as gr
import torch, random, os
from datetime import datetime

LOCAL_DIR = '/content/generated_images'
os.makedirs(LOCAL_DIR, exist_ok=True)

# ── Resolutions ──────────────────────────────────────────────────
# SDXL native resolution is 1024×1024.
# 896×1152 is ideal for portrait: close to native, great for people.
RESOLUTIONS = {
    'Portrait   896 × 1152  📱  (best for people)': (896, 1152),
    'Square    1024 × 1024  ⬜  (general use)'    : (1024, 1024),
    'Landscape 1152 × 896   🖼️  (wide shots)'     : (1152, 896),
    'Tall       832 × 1216  📏  (full body)'       : (832, 1216),
}

# ── Default prompts ──────────────────────────────────────────────
# LUSTIFY v2 understands natural language — write like you're
# describing a photograph. Photography terms greatly improve quality.
DEFAULT_POS = (
    'RAW photo, a 25-year-old woman, natural skin texture, '
    'soft cinematic lighting, shallow depth of field, '
    'shot on Canon EOS R5, 85mm lens, bokeh background, '
    'hyperrealistic, high detail, 8K'
)
DEFAULT_NEG = (
    'cartoon, anime, illustration, painting, drawing, art, '
    'cgi, 3d render, bad anatomy, bad hands, extra fingers, '
    'missing fingers, deformed, disfigured, ugly, blurry, '
    'low quality, worst quality, watermark, signature, text, '
    'poorly drawn face, cloned face, extra limbs, mutilated'
)

# ── Generation function ──────────────────────────────────────────
def generate(pos, neg, res_label, steps, cfg, seed_val, save_drive,
             progress=gr.Progress()):

    seed = random.randint(0, 2**32 - 1) if int(seed_val) == -1 else int(seed_val)
    w, h = RESOLUTIONS[res_label]
    gen  = torch.Generator(device='cuda').manual_seed(seed)

    progress(0.05, desc='⏳ Starting generation...')

    def on_step(p, i, t, kwargs):
        pct = 0.05 + 0.90 * ((i + 1) / int(steps))
        progress(min(pct, 0.95), desc=f'🔄 Step {i+1} / {int(steps)}')
        return kwargs

    try:
        out = pipe(
            prompt=pos,
            negative_prompt=neg,
            width=w,
            height=h,
            num_inference_steps=int(steps),
            guidance_scale=float(cfg),
            generator=gen,
            callback_on_step_end=on_step,
        )
    except RuntimeError as e:
        torch.cuda.empty_cache()
        if 'out of memory' in str(e).lower():
            return (
                None,
                '❌ VRAM out of memory!',
                'Fix: lower Steps to 20, switch to Portrait resolution, '
                'then run the VRAM Cleanup section below and try again.'
            )
        return None, f'❌ Error: {e}', ''

    progress(1.0, desc='✅ Done!')
    img = out.images[0]

    # Save locally
    ts   = datetime.now().strftime('%Y%m%d_%H%M%S')
    name = f'realistic_{ts}_seed{seed}.png'
    img.save(os.path.join(LOCAL_DIR, name))
    msg  = f'💾 Saved: {LOCAL_DIR}/{name}'

    # Optionally save to Google Drive
    drive_dir = '/content/drive/MyDrive/AI_Realistic_Images'
    if save_drive:
        if os.path.exists('/content/drive/MyDrive'):
            os.makedirs(drive_dir, exist_ok=True)
            img.save(os.path.join(drive_dir, name))
            msg += f'\n📁 Drive: {drive_dir}/{name}'
        else:
            msg += (
                '\n⚠️  Drive not mounted. Run this in a new cell first:\n'
                '   from google.colab import drive; drive.mount("/content/drive")'
            )

    info = f'📐 {w}×{h}   🔢 Steps: {int(steps)}   ⚙️ CFG: {cfg}   🎲 Seed: {seed}'
    return img, info, msg


# ── Gradio UI ─────────────────────────────────────────────────────
CSS = """
.gen-btn {
    background: linear-gradient(135deg, #1d4ed8, #0f766e) !important;
    color: #fff !important;
    font-size: 17px !important;
    font-weight: 700 !important;
    border: none !important;
    border-radius: 10px !important;
    min-height: 52px !important;
}
.gen-btn:hover { opacity: 0.88 !important; }
"""

with gr.Blocks(
    title='📸 Realistic AI Generator',
    theme=gr.themes.Soft(primary_hue='blue', secondary_hue='teal'),
    css=CSS
) as demo:

    gr.Markdown("""
    # 📸 Realistic NSFW Image Generator
    **Model:** LUSTIFY v2.0 (SDXL) &nbsp;|&nbsp; **GPU:** T4 15 GB &nbsp;|&nbsp;
    **Speed:** ~60–90 sec/image &nbsp;|&nbsp; **No login required ✅**

    > This model understands **natural language** — write prompts like describing a photograph.
    > Use photography terms (camera, lens, lighting) for the most realistic results.
    """)

    with gr.Row():

        # ── Left: Controls ─────────────────────────────────────
        with gr.Column(scale=5):
            pos_box = gr.Textbox(
                label='✏️  Positive Prompt  (describe what you want)',
                value=DEFAULT_POS,
                lines=4,
                placeholder='RAW photo, a woman, natural lighting, ...'
            )
            neg_box = gr.Textbox(
                label='🚫  Negative Prompt  (what to avoid)',
                value=DEFAULT_NEG,
                lines=3,
            )
            res_radio = gr.Radio(
                label='📐  Resolution',
                choices=list(RESOLUTIONS.keys()),
                value='Portrait   896 × 1152  📱  (best for people)',
            )
            with gr.Row():
                steps_sl = gr.Slider(
                    label='🔢 Steps  (25–30 recommended)',
                    minimum=15, maximum=40, step=1, value=28
                )
                cfg_sl = gr.Slider(
                    label='⚙️ CFG Scale  (4–7 recommended)',
                    minimum=1.0, maximum=10.0, step=0.5, value=5.0
                )
            seed_box = gr.Number(
                label='🎲 Seed  (−1 = random each time)',
                value=-1, precision=0
            )
            drive_cb = gr.Checkbox(
                label='📁 Also save to Google Drive',
                value=False,
                info='Mount Drive first: from google.colab import drive; drive.mount("/content/drive")'
            )
            gen_btn = gr.Button(
                '🚀  Generate Image',
                elem_classes='gen-btn',
                variant='primary'
            )

            gr.Markdown('### 💡 Example prompts (click to load):')
            gr.Examples(
                label='',
                examples=[
                    [
                        'RAW photo, a 28-year-old woman with long brunette hair, blue eyes, natural makeup, '
                        'white linen dress, standing on a balcony at golden hour, soft warm lighting, '
                        'shallow depth of field, shot on Sony A7 III, 85mm f/1.4, bokeh, hyperrealistic, 8K',
                        DEFAULT_NEG
                    ],
                    [
                        'RAW photo, a fit 30-year-old woman, short red hair, green eyes, '
                        'black lingerie, bedroom setting, soft window light, '
                        'shot on Leica T, intimate pose, skin texture detail, nsfw, hyperrealistic',
                        DEFAULT_NEG
                    ],
                    [
                        'RAW photo, a 25-year-old woman, blonde hair, hazel eyes, '
                        'bikini, poolside, summer afternoon, sunlight, wet skin, '
                        'shot on Canon EOS 5D, lifestyle photography, hyperrealistic, 8K',
                        DEFAULT_NEG
                    ],
                    [
                        'RAW photo, a 35-year-old man, short dark hair, strong jaw, stubble, '
                        'casual white t-shirt, outdoor cafe, natural light, candid moment, '
                        'shot on Fujifilm X-T4, street photography, sharp focus, hyperrealistic',
                        DEFAULT_NEG
                    ],
                    [
                        'RAW photo, a couple in their 20s, embracing, sunset beach, '
                        'golden hour backlighting, romantic atmosphere, warm tones, '
                        'shot on Nikon Z6, 50mm, cinematic, hyperrealistic, 8K',
                        DEFAULT_NEG
                    ],
                    [
                        'glamour photography, a 27-year-old woman, dark skin, natural afro hair, '
                        'red evening gown, studio lighting, dramatic shadows, '
                        'shot on Polaroid SX-70, fashion editorial, hyperrealistic',
                        DEFAULT_NEG
                    ],
                ],
                inputs=[pos_box, neg_box],
            )

        # ── Right: Output ───────────────────────────────────────
        with gr.Column(scale=5):
            out_img  = gr.Image(label='🖼️  Generated Image', type='pil', height=570)
            info_box = gr.Textbox(label='ℹ️  Generation Info', interactive=False, lines=1)
            save_box = gr.Textbox(label='💾  Save Status',     interactive=False, lines=2)

    gen_btn.click(
        fn=generate,
        inputs=[pos_box, neg_box, res_radio, steps_sl, cfg_sl, seed_box, drive_cb],
        outputs=[out_img, info_box, save_box],
    )

    # ── Prompting Guide ────────────────────────────────────────
    with gr.Accordion('📖  Prompting Guide — How to Get the Best Results', open=False):
        gr.Markdown("""
        ## Prompting Guide for LUSTIFY v2.0

        This model excels at **natural language** prompts — write like you're briefing a photographer.
        It also accepts Danbooru tags if you prefer that style.

        ---

        ### 📸 Photography terms that dramatically improve realism:

        **Prefix your prompt with one of these:**
        ```
        RAW photo,        analog photo,       glamour photography,
        street photography,  editorial photo,   lifestyle photography
        ```

        **Camera bodies:**
        `shot on Canon EOS R5`  `shot on Sony A7 III`  `shot on Nikon Z6`
        `shot on Leica T`  `shot on Fujifilm X-T4`  `shot on Polaroid SX-70`

        **Lenses / depth of field:**
        `85mm f/1.4`  `50mm lens`  `35mm f/1.8`  `shallow depth of field`  `bokeh`

        **Lighting:**
        `soft cinematic lighting`  `golden hour`  `soft window light`
        `dramatic shadows`  `studio lighting`  `neon lighting`
        `warm golden hour lighting`  `radiant god rays`

        **Film / quality tags:**
        `Ilford HP5 Plus`  `Lomochrome color film`  `Fujicolor Pro`
        `hyperrealistic`  `8K`  `high detail`  `sharp focus`

        ---

        ### 👤 Describing people:
        Be specific: age, hair color/length, eye color, ethnicity, body type, expression.
        Example: `a 27-year-old woman, long dark hair, green eyes, athletic build, smiling`

        ### 🏞️ Settings & scenes:
        `bedroom`  `poolside`  `beach`  `outdoor cafe`  `studio`  `forest`
        `hotel room`  `balcony`  `sunlit room`  `dark urban alley`

        ### 🔞 Content level:
        | Add to prompt | Effect |
        |---------------|--------|
        | *(nothing)* | SFW / tasteful |
        | `suggestive` | Revealing but no nudity |
        | `nsfw` | Adult, partial nudity |
        | `explicit, nude` | Fully explicit |

        ---

        ### ⚙️ Settings guide:
        | Setting | Best value | Notes |
        |---------|------------|-------|
        | Steps | 25–30 | 28 is the sweet spot |
        | CFG | 4–6 | **Lower than anime models!** — 5 is ideal, don't go above 7 |
        | Seed | -1 | Random each run |
        | Resolution | 896×1152 | Best for portrait/people |

        > ⚠️ **CFG note:** Realistic models need lower CFG than anime models.
        > CFG above 7 makes skin look waxy and over-processed. Stay at 4–6.

        ### 🔁 Reproducing a result:
        Copy the **Seed** from ℹ️ Info → paste into Seed field → same prompt = same image.
        """)

    # ── Utilities ─────────────────────────────────────────────
    with gr.Accordion('🧹  VRAM Cleanup  (run if you get Out of Memory errors)', open=False):
        gr.Markdown("""
        If you get an **Out of Memory** error, run this in a new Colab cell:
        ```python
        import torch, gc
        del pipe
        gc.collect()
        torch.cuda.empty_cache()
        print('VRAM cleared — re-run Cell 4 to reload the model')
        ```
        """)

    with gr.Accordion('⬇️  Download All Images as ZIP', open=False):
        gr.Markdown("""
        Run this in a new Colab cell to zip and download all your images:
        ```python
        import zipfile, os
        from google.colab import files
        DIR = '/content/generated_images'
        imgs = [f for f in os.listdir(DIR) if f.endswith('.png')]
        if imgs:
            with zipfile.ZipFile('/content/realistic_images.zip', 'w') as zf:
                for f in imgs: zf.write(f'{DIR}/{f}', f)
            files.download('/content/realistic_images.zip')
            print(f'Downloaded {len(imgs)} images!')
        else:
            print('No images yet!')
        ```
        """)

print('🚀 Launching...')
demo.launch(share=True, debug=False)